<a href="https://colab.research.google.com/github/flavianamsamorim/Glass-Container-Industry-Problem-New-Furnace-Installation-GCIP-NF-/blob/main/Glass_Container_Industry_Problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Glass Container Industry Problem – New Furnace Installation (GCIP-NF)**

This section presents a mixed-integer mathematical programming model for the Glass Container Industry Problem regarding a New Furnace installation (GCIP-NF).

***Parameters***

* m: Machines available (m = 1, ..., M)
* i: Containers to be manufactured (i = 1, ..., I)
* a: Annual time horizon (a = 1, ..., A)

* NSₘ: Number of sections of machine m
* TGₘ: Type of gob of machine m
* ACᵢₘ: 1 if container i is accepted by machine m, 0 otherwise
* Cₘ: Cost to install machine m (cifrão)
* Dᵢₐ: Demand  of container i in period a (ton)
* Wᵢ: Weight of container i (ton)
* Rᵢ: Cavity rate of container i (bottles/year)
* M̄: Maximum number of machines supported by the furnace
* CF: Cost to install melting capacity on the furnace ($/ton)
* ηₘ: Efficiency of machine m

***Decision Variables***

* KF: Required melting capacity of the furnace (ton)

* Qᵢₘₐ: Lot size of container i produced on machine m in period a (ton)

* Fᵢₘₐ: Fraction of time spent producing container i on machine m in year a

* Ȳₘ: 1 if machine m is installed, 0 otherwise

**Objective Function**

Minimize total furnace and machine installation cost:

Minimize:

f(KF, Ȳ) = CF · KF + Σₘ Cₘ · Ȳₘ

*Constraints Machine Limit*

The number of installed machines cannot exceed furnace capacity:

Σₘ Ȳₘ ≤ M̄

*Production Feasibility*

Production is allowed only if the machine is installed and compatible:

Fᵢₘₐ ≤ Ȳₘ · ACᵢₘ for all (i, m, a)

*Annual Machine Time Capacity*

Each machine has at most one year of production time:

Σᵢ Fᵢₘₐ ≤ 1 for all (m, a)

*Production Calculation*

The production in tons is calculated as:

Qᵢₘₐ = Fᵢₘₐ · (Rᵢ · Wᵢ · NSₘ · TGₘ · ηₘ)

*Demand Satisfaction*

Cumulative production must meet cumulative demand:

Σ_{τ=1..a} Σₘ Qᵢₘτ ≥ Σ_{τ=1..a} Dᵢτ for all (i, a)

*Furnace Capacity*

Total annual production must not exceed furnace capacity:

Σᵢ Σₘ Qᵢₘₐ ≤ KF for all (a)

*Variable Domains*

Qᵢₘₐ ≥ 0
Fᵢₘₐ ≥ 0
KF ≥ 0
Ȳₘ ∈ {0,1}

In [6]:
import pulp as pl
import numpy as np
import re
import time  # <--- Added to compute execution time
from google.colab import files


# ==========================================================
# 1️⃣ ROBUST .dat READER (AMPL-style format)
# ==========================================================


def read_instance(file_path):
    with open(file_path, 'r') as f:
        text = f.read()

    text = text.replace('\t', ' ')
    text = text.replace(';', '')

    data = {}
    scalars = ['T', 'M', 'I', 'M_max', 'CF']

    for s in scalars:
        pattern = rf"{s}\s*=\s*([0-9\.]+)"
        match = re.search(pattern, text)
        if match:
            value = float(match.group(1))
            if value.is_integer():
                value = int(value)
            data[s] = value

    def parse_vector(name):
        pattern = rf"{name}\s*=\s*\[(.*?)\]"
        match = re.search(pattern, text, re.DOTALL)
        if match:
            raw = match.group(1)
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", raw)
            return np.array([float(x) for x in nums])
        return None

    def parse_matrix(name):
        start = text.find(f"{name}")
        if start == -1: return None
        start = text.find('[', start)
        bracket_count = 0
        end = start
        while end < len(text):
            if text[end] == '[': bracket_count += 1
            elif text[end] == ']':
                bracket_count -= 1
                if bracket_count == 0: break
            end += 1
        block = text[start:end+1]
        rows = re.findall(r"\[(.*?)\]", block, re.DOTALL)
        matrix = []
        for r in rows:
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", r)
            matrix.append([float(x) for x in nums])
        return np.array(matrix)

    vector_names = ['NSm','TGm','Cm','Nm','Wp','Rp']
    for v in vector_names: data[v] = parse_vector(v)
    data['ACpm'] = parse_matrix('ACpm')
    data['Dpt']  = parse_matrix('Dpt')

    return data


# ==========================================================
# 2️⃣ COMPLETE GCIP MODEL
# ==========================================================


def solve_GCIP(instance_file):
    # Start clock
    start_time = time.time()

    data = read_instance(instance_file)

    I = int(data['I'])
    M = int(data['M'])
    T = int(data['T'])
    M_max = int(data['M_max'])
    CF = float(data['CF'])

    NSm = data['NSm']
    TGm = data['TGm']
    Cm  = data['Cm']
    Nm  = data['Nm']
    Wp  = data['Wp']
    Rp  = data['Rp']
    ACpm = data['ACpm']
    Dpt  = data['Dpt']

    products = range(I)
    machines = range(M)
    periods  = range(T)

    model = pl.LpProblem("GCIP", pl.LpMinimize)

    Y = pl.LpVariable.dicts("Install", machines, cat="Binary")
    F = pl.LpVariable.dicts("TimeFrac", [(i,m,t) for i in products for m in machines for t in periods], lowBound=0)
    Q = pl.LpVariable.dicts("Production", [(i,m,t) for i in products for m in machines for t in periods], lowBound=0)
    KF = pl.LpVariable("FurnaceCapacity", lowBound=0)

    model += CF * KF + pl.lpSum(Cm[m] * Y[m] for m in machines)
    model += pl.lpSum(Y[m] for m in machines) <= M_max

    for i in products:
        for m in machines:
            for t in periods:
                model += F[(i,m,t)] <= Y[m] * ACpm[i][m]

    for m in machines:
        for t in periods:
            model += pl.lpSum(F[(i,m,t)] for i in products) <= 1

    MIN_PER_YEAR = 525600
    for i in products:
      for m in machines:
        for t in periods:
            model += (Q[(i,m,t)] == F[(i,m,t)] * Rp[i] * Wp[i] * NSm[m] * TGm[m] * Nm[m] * MIN_PER_YEAR, f"Prod_i{i}_m{m}_t{t}")

    for i in products:
        for t in range(T):
              model += (pl.lpSum(Q[(i,m,tt)] for tt in range(t+1) for m in machines) >= sum(Dpt[i][tt] for tt in range(t+1)), f"Demand_cumulative_i{i}_upto_{t}")

    for t in periods:
        model += pl.lpSum(Q[(i,m,t)] for i in products for m in machines) <= KF

    solver = pl.PULP_CBC_CMD(msg=1)
    model.solve(solver)

    # End clock
    end_time = time.time()
    execution_time = end_time - start_time

    # ==========================================================
    # DETAILED RESULTS
    # ==========================================================

    print("\nStatus:", pl.LpStatus[model.status])
    print(f"Execution time: {execution_time:.2f} seconds") # <--- Clock output
    print("Total Cost:", pl.value(model.objective))
    print("Furnace Capacity:", pl.value(KF))

    installed = [m for m in machines if pl.value(Y[m]) > 0.5]
    print("Installed Machines:", installed)

    for m in installed:
        print(f"\n--- Machine {m} ---")
        print(f"Installation Cost: {Cm[m]}")
        print(f"NSm: {NSm[m]}")
        print(f"TGm: {TGm[m]}")
        print(f"Nm (efficiency): {Nm[m]:.4f}")
        total_machine_prod = 0
        for t in periods:
            period_prod = sum(pl.value(Q[(i,m,t)]) for i in products)
            total_machine_prod += period_prod
        print(f"Total production of machine {m}: {total_machine_prod:.2f}")

# ==========================================================
# EXECUTION VIA UPLOAD (Colab/Jupyter)
# ==========================================================


if __name__ == "__main__":
    print("Select the .dat file")
    uploaded = files.upload()
    if uploaded:
        instance_path = list(uploaded.keys())[0]
        solve_GCIP(instance_path)

Select the .dat file


Saving RealInstance.dat to RealInstance.dat

Status: Optimal
Execution time: 0.13 seconds
Total Cost: 3234065.5
Furnace Capacity: 2389.21
Installed Machines: [0, 8]

--- Machine 0 ---
Installation Cost: 950000.0
NSm: 16.0
TGm: 1.0
Nm (efficiency): 0.9150
Total production of machine 0: 1833.46

--- Machine 8 ---
Installation Cost: 970000.0
NSm: 8.0
TGm: 2.0
Nm (efficiency): 0.8990
Total production of machine 8: 555.75


In [11]:
import numpy as np
import heapq
import time
import random
import re
from google.colab import files


# =============================================================================
# 1. INSTANCE CLASS WITH RESILIENT PARSER
# =============================================================================
class GCIP_Instance:
    def __init__(self, data_content):
        data = self._parse_dat(data_content)
        try:
            self.A = int(data['T'])
            self.M = int(data['M'])
            self.I = int(data['I'])
            self.M_bar = int(data['M_max'])
            self.CF = float(data['CF'])
            self.NS = np.array(data['NSm'])
            self.TG = np.array(data['TGm'])
            self.Cm = np.array(data['Cm'])
            self.eta = np.array(data['Nm'])
            self.W = np.array(data['Wp'])
            self.R_min = np.array(data['Rp'])
            self.AC = np.array(data['ACpm']).reshape((self.I, self.M))
            self.D = np.array(data['Dpt']).reshape((self.I, self.A))
            self.T = np.zeros((self.I, self.M))
            for i in range(self.I):
                for m in range(self.M):
                    self.T[i, m] = self.R_min[i] * self.W[i] * self.NS[m] * self.TG[m] * self.eta[m] * 525600
        except KeyError as e:
            print(f"CRITICAL ERROR: Variable {e} not found in .dat file")
            raise

    def _parse_dat(self, content):
        parsed = {}
        content = re.sub(r'#.*', '', content)
        content = re.sub(r'/\*.*?\*/', '', content, flags=re.DOTALL)
        content = content.replace(';', ' ')
        pattern = re.compile(r'(\w+)\s*=\s*([^=]+?)(?=\s*\w+\s*=|\s*$)')
        for match in pattern.finditer(content):
            name = match.group(1).strip()
            val_raw = match.group(2).strip()
            if '[' in val_raw:
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", val_raw)
                parsed[name] = [float(x) for x in nums]
            else:
                try: parsed[name] = float(val_raw)
                except ValueError: parsed[name] = val_raw
        return parsed


# =============================================================================
# 2. GREEDY FILTER HEURISTIC (GF)
# =============================================================================
def greedy_filter(chromosome, inst, detailed=False):
    Y = np.array(chromosome)
    penalty = 1e20
    if np.sum(Y) > inst.M_bar or np.sum(Y) == 0:
        return (penalty, None) if detailed else penalty
    for i in range(inst.I):
        if np.any(inst.D[i, :] > 1e-3):
            if not any((Y[m] == 1 and inst.AC[i, m] == 1) for m in range(inst.M)):
                return (penalty, None) if detailed else penalty
    S = inst.D.copy().astype(float)
    KF = 0.0
    prod_m = np.zeros(inst.M)
    util_m = np.zeros((inst.M, inst.A))
    for a in range(inst.A - 1, -1, -1):
        K_period = 0.0
        R_ma = np.ones(inst.M)
        m_order = np.argsort([np.sum(inst.AC[:, m]) for m in range(inst.M)])
        for m in m_order:
            if Y[m] == 1:
                i_order = np.argsort(inst.T[:, m])[::-1]
                for i in i_order:
                    if inst.AC[i, m] == 1 and S[i, a] > 0 and R_ma[m] > 1e-9:
                        pa = R_ma[m] * inst.T[i, m]
                        q = min(pa, S[i, a])
                        S[i, a] -= q
                        R_ma[m] -= (q / inst.T[i, m])
                        K_period += q
                        prod_m[m] += q
                util_m[m, a] = 1.0 - R_ma[m]
        KF = max(KF, K_period)
        if np.sum(S[:, a]) > 1e-2:
            return (penalty, None) if detailed else penalty
    total_cost = (inst.CF * KF) + np.sum(inst.Cm * Y)
    return (total_cost, {"KF": KF, "prod": prod_m, "util": util_m}) if detailed else total_cost


# =============================================================================
# 3. MPGAt (GENETIC ALGORITHM)
# =============================================================================
class Island:
    def __init__(self, size, inst):
        self.size, self.inst, self.population = size, inst, []
    def initialize(self):
        while len(self.population) < self.size:
            chrom = np.zeros(self.inst.M, dtype=int)
            idx = np.random.choice(range(self.inst.M), self.inst.M_bar, replace=False)
            chrom[idx] = 1
            self.population.append((greedy_filter(chrom, self.inst), list(chrom)))
        heapq.heapify(self.population)
    def add(self, child):
        worst = heapq.nlargest(1, self.population)[0]
        if child[0] < worst[0]:
            self.population.remove(worst); heapq.heappush(self.population, child)


def run_mpga_t(inst, n_islands=10, isl_size=60, epochs=600):
    islands = [Island(isl_size, inst) for _ in range(n_islands)]
    for isl in islands: isl.initialize()
    for epoch in range(epochs):
        for isl in islands:
            for _ in range(isl_size // 2):
                p1, p2 = isl.population[0], random.choice(isl.population)
                child = [p1[1][g] if random.random() < 0.6 else p2[1][g] for g in range(inst.M)]
                if random.random() < 0.3:
                    m_idx = random.randint(0, inst.M - 1); child[m_idx] = 1 - child[m_idx]
                isl.add((greedy_filter(child, inst), child))
            heapq.heapify(isl.population)
        if epoch % 50 == 0:
            for i in range(n_islands): islands[(i+1)%n_islands].add(islands[i].population[0])
    return min([isl.population[0] for isl in islands], key=lambda x: x[0])


# =============================================================================
# 4. MAIN EXECUTION (WITH TOTAL TIME CLOCK)
# =============================================================================
def solve_GCIP(instance_path):
    with open(instance_path, 'r') as f:
        content = f.read()

    inst = GCIP_Instance(content)
    print(f"--- Processing Instance: {instance_path} ---")

    # Start timing
    start_time = time.time()

    best_fit, best_chrom = run_mpga_t(inst)
    cost, det = greedy_filter(best_chrom, inst, detailed=True)

    # End timing
    total_time = time.time() - start_time

    if det:
        maquinas = [m for m, v in enumerate(best_chrom) if v == 1]
        print(f"\nStatus: Optimal (Metaheuristic: MPGAt-GF)")
        print(f"Total Cost: {cost:.2f}")
        print(f"Furnace Capacity (KF): {det['KF']:.2f}")
        print(f"Installed Machines: {maquinas}")

        for m in maquinas:
            print(f"\n--- Machine {m} ---")
            print(f"Installation Cost: {inst.Cm[m]:.1f}")
            print(f"NSm: {inst.NS[m]} | TGm: {inst.TG[m]} | Nm: {inst.eta[m]:.4f}")
            print(f"Total production on machine: {det['prod'][m]:.2f}")

        #print("\n================ ACTUAL UTILIZATION ================")
        for m in maquinas:
            for a in range(inst.A):
                u = det['util'][m, a]
                if u > 0:
                    #print(f"Machine {m} - period {a} - utilization: {u:.4f}")

                    print("\n" + "="*40)
        print(f"TOTAL TIME SPENT: {total_time:.4f} seconds")
        print("="*40)
    else:
        print("\nERROR: No feasible solution found.")
        print(f"Search time until failure: {total_time:.4f} seconds")

# ==========================================================
# EXECUTION VIA UPLOAD (Colab/Jupyter)
# ==========================================================

if __name__ == "__main__":
    uploaded = files.upload()
    if uploaded:
        solve_GCIP(list(uploaded.keys())[0])

Saving RealInstance.dat to RealInstance (4).dat
--- Processing Instance: RealInstance (4).dat ---

Status: Optimal (Metaheuristic: MPGAt-GF)
Total Cost: 3234065.50
Furnace Capacity (KF): 2389.21
Installed Machines: [0, 8]

--- Machine 0 ---
Installation Cost: 950000.0
NSm: 16.0 | TGm: 1.0 | Nm: 0.9150
Total production on machine: 1939.48

--- Machine 8 ---
Installation Cost: 970000.0
NSm: 8.0 | TGm: 2.0 | Nm: 0.8990
Total production on machine: 449.73


TOTAL TIME SPENT: 33.3851 seconds


In [14]:
import numpy as np
import time
import random
import re
from google.colab import files

# =============================================================================
# 1. INSTANCE CLASS WITH RESILIENT PARSER
# =============================================================================
class GCIP_Instance:
    def __init__(self, data_content):
        data = self._parse_dat(data_content)
        try:
            self.A = int(data['T'])              # Number of periods
            self.M = int(data['M'])              # Number of candidate machines
            self.I = int(data['I'])              # Number of product types
            self.M_bar = int(data['M_max'])      # Max machines supported by furnace
            self.CF = float(data['CF'])          # Furnace capacity cost per ton
            self.NS = np.array(data['NSm'])      # Number of sections per machine
            self.TG = np.array(data['TGm'])      # Gob type per machine
            self.Cm = np.array(data['Cm'])       # Installation cost per machine
            self.eta = np.array(data['Nm'])      # Efficiency per machine
            self.W = np.array(data['Wp'])        # Weight per product type
            self.R_min = np.array(data['Rp'])    # Cavity rate (bottles/min)
            self.AC = np.array(data['ACpm']).reshape((self.I, self.M)) # Compatibility matrix
            self.D = np.array(data['Dpt']).reshape((self.I, self.A))   # Demand per product/period

            # Theoretical production capacity (T_im)
            self.T_capacity = np.zeros((self.I, self.M))
            for i in range(self.I):
                for m in range(self.M):
                    # Formula: Rate * Weight * Sections * Gobs * Efficiency * Minutes in a year
                    self.T_capacity[i, m] = self.R_min[i] * self.W[i] * self.NS[m] * self.TG[m] * self.eta[m] * 525600
        except KeyError as e:
            print(f"CRITICAL ERROR: Variable {e} was not found in the .dat file")
            raise

    def _parse_dat(self, content):
        parsed = {}
        # Remove comments (single line and block)
        content = re.sub(r'#.*', '', content)
        content = re.sub(r'/\*.*?\*/', '', content, flags=re.DOTALL)
        content = content.replace(';', ' ')

        # Regex pattern to match variable assignments
        pattern = re.compile(r'(\w+)\s*=\s*([^=]+?)(?=\s*\w+\s*=|\s*$)')
        for match in pattern.finditer(content):
            name = match.group(1).strip()
            val_raw = match.group(2).strip()
            if '[' in val_raw:
                # Extract numeric values from lists/matrices
                nums = re.findall(r"[-+]?\d*\.\d+|\d+", val_raw)
                parsed[name] = [float(x) for x in nums]
            else:
                try: parsed[name] = float(val_raw)
                except ValueError: parsed[name] = val_raw
        return parsed

# =============================================================================
# 2. GREEDY FILTER (GF) HEURISTIC
# =============================================================================
def greedy_filter(chromosome, inst, detailed=False):
    Y = np.array(chromosome)
    penalty = 1e20

    # Check basic feasibility (Machine limit and at least one machine active)
    if np.sum(Y) > inst.M_bar or np.sum(Y) == 0:
        return (penalty, None) if detailed else penalty

    # Check if demand for each product can be met by active compatible machines
    for i in range(inst.I):
        if np.any(inst.D[i, :] > 1e-3):
            if not any((Y[m] == 1 and inst.AC[i, m] == 1) for m in range(inst.M)):
                return (penalty, None) if detailed else penalty

    S = inst.D.copy().astype(float) # Remaining demand to satisfy
    KF = 0.0                        # Required furnace capacity
    prod_m = np.zeros(inst.M)       # Total production per machine
    util_m = np.zeros((inst.M, inst.A)) # Utilization rate per machine/period

    # Iterate backwards through periods to calculate furnace requirement
    for a in range(inst.A - 1, -1, -1):
        K_period = 0.0
        R_ma = np.ones(inst.M)      # Remaining time resource for machine m in period a

        # Order machines by compatibility (least versatile first)
        m_order = np.argsort([np.sum(inst.AC[:, m]) for m in range(inst.M)])
        for m in m_order:
            if Y[m] == 1:
                # Order products by production rate on machine m (highest first)
                i_order = np.argsort(inst.T_capacity[:, m])[::-1]
                for i in i_order:
                    if inst.AC[i, m] == 1 and S[i, a] > 0 and R_ma[m] > 1e-9:
                        pa = R_ma[m] * inst.T_capacity[i, m] # Potential production
                        q = min(pa, S[i, a])                 # Actual production allocated
                        S[i, a] -= q
                        R_ma[m] -= (q / inst.T_capacity[i, m])
                        K_period += q
                        prod_m[m] += q
                util_m[m, a] = 1.0 - R_ma[m]

        KF = max(KF, K_period)
        if np.sum(S[:, a]) > 1e-2: # If demand remains unsatisfied
            return (penalty, None) if detailed else penalty

    total_cost = (inst.CF * KF) + np.sum(inst.Cm * Y)
    return (total_cost, {"KF": KF, "prod": prod_m, "util": util_m}) if detailed else total_cost

# =============================================================================
# 3. SPARROW SEARCH ALGORITHM (SSA)
# =============================================================================
class SSA_Optimizer:
    def __init__(self, inst, pop_size=50, time_limit=60):
        self.inst = inst
        self.pop_size = pop_size
        self.time_limit = time_limit
        self.p_percent = 0.2  # Percentage of producers
        self.v_percent = 0.15 # Percentage of vigilants
        self.population = []
        self.best_sol = {'chrom': None, 'fit': float('inf')}

    def initialize(self):
        for _ in range(self.pop_size):
            chrom = np.zeros(self.inst.M, dtype=int)
            # Randomly activate between 1 and M_bar machines
            idx = np.random.choice(range(self.inst.M), random.randint(1, self.inst.M_bar), replace=False)
            chrom[idx] = 1
            fit = greedy_filter(chrom, self.inst)
            ind = {'chrom': chrom, 'fit': fit}
            self.population.append(ind)
            if fit < self.best_sol['fit']:
                self.best_sol = {'chrom': chrom.copy(), 'fit': fit}

    def solve(self):
        self.initialize()
        start_time = time.time()

        while (time.time() - start_time) < self.time_limit:
            self.population.sort(key=lambda x: x['fit'])
            num_producers = int(self.pop_size * self.p_percent)
            num_vigilants = int(self.pop_size * self.v_percent)
            producers = self.population[:num_producers]
            followers = self.population[num_producers:]

            # Update Producers
            for ind in producers:
                new_chrom = ind['chrom'].copy()
                if random.random() < 0.8: # Local search: bit flip
                    idx = random.randint(0, self.inst.M - 1)
                    new_chrom[idx] = 1 - new_chrom[idx]
                else: # Global reset
                    new_chrom = np.zeros(self.inst.M, dtype=int)
                    idx = np.random.choice(range(self.inst.M), random.randint(1, self.inst.M_bar), replace=False)
                    new_chrom[idx] = 1
                ind['chrom'], ind['fit'] = self.evaluate_new(new_chrom)

            # Update Followers
            worst_producer = producers[0]
            for ind in followers:
                new_chrom = ind['chrom'].copy()
                if random.random() > 0.5: # Follow the best (masking)
                    mask = np.random.rand(self.inst.M) > 0.5
                    new_chrom[mask] = worst_producer['chrom'][mask]
                else: # Random perturbation
                    idx = random.randint(0, self.inst.M - 1)
                    new_chrom[idx] = 1 - new_chrom[idx]
                ind['chrom'], ind['fit'] = self.evaluate_new(new_chrom)

            # Update Vigilants (Anti-predation behavior)
            vigilant_indices = np.random.choice(range(self.pop_size), num_vigilants, replace=False)
            for idx in vigilant_indices:
                ind = self.population[idx]
                new_chrom = ind['chrom'].copy()
                if ind['fit'] > self.best_sol['fit']:
                    mask = np.random.rand(self.inst.M) > 0.3
                    new_chrom[mask] = self.best_sol['chrom'][mask]
                else:
                    i = random.randint(0, self.inst.M - 1)
                    new_chrom[i] = 1 - new_chrom[i]
                ind['chrom'], ind['fit'] = self.evaluate_new(new_chrom)

            # Update Global Best
            for ind in self.population:
                if ind['fit'] < self.best_sol['fit']:
                    self.best_sol = {'chrom': ind['chrom'].copy(), 'fit': ind['fit']}

        return self.best_sol['fit'], self.best_sol['chrom']

    def evaluate_new(self, chrom):
        # Repair mechanism: ensure machine limit is respected
        if np.sum(chrom) > self.inst.M_bar:
            active_indices = np.where(chrom == 1)[0]
            to_deactivate = np.random.choice(active_indices, np.sum(chrom) - self.inst.M_bar, replace=False)
            chrom[to_deactivate] = 0
        fit = greedy_filter(chrom, self.inst)
        return chrom, fit

# =============================================================================
# 4. EXECUTION WITH TOTAL TIME CLOCK
# =============================================================================
def solve_GCIP(instance_path):
    with open(instance_path, 'r') as f:
        content = f.read()

    inst = GCIP_Instance(content)
    print(f"\n--- SSA Optimizer | Time Limit: 60s ---")

    # --- CLOCK START ---
    start_process = time.time()

    ssa = SSA_Optimizer(inst, pop_size=100, time_limit=60)
    best_fit, best_chrom = ssa.solve()

    # Re-run filter for final detailed results
    cost, det = greedy_filter(best_chrom, inst, detailed=True)

    # --- CLOCK END ---
    end_process = time.time()
    execution_time = end_process - start_process

    if det:
        print(f"\nStatus: Optimal (Metaheuristic-SSA)")
        print(f"Execution time: {execution_time:.4f} seconds")
        print(f"Total Cost: {cost:.2f}")
        print(f"Furnace Capacity (KF): {det['KF']:.2f}")

        installed_machines = [m for m, v in enumerate(best_chrom) if v == 1]
        print(f"Installed Machines: {installed_machines}")

        for m in installed_machines:
            print(f"\n--- Machine {m} ---")
            print(f"Installation Cost: {inst.Cm[m]}")
            print(f"Sections (NSm): {inst.NS[m]}")
            print(f"Gob Type (TGm): {inst.TG[m]}")
            print(f"Efficiency (Nm): {inst.eta[m]:.4f}")
            print(f"Total production on machine {m}: {det['prod'][m]:.2f}")

# ==========================================================
# EXECUTION VIA UPLOAD (Colab/Jupyter)
# ==========================================================

if __name__ == "__main__":
    uploaded = files.upload()
    if uploaded:
        instance_name = list(uploaded.keys())[0]
        solve_GCIP(instance_name)

Saving RealInstance.dat to RealInstance (7).dat

--- SSA Optimizer | Time Limit: 60s ---

Status: Optimal (Metaheuristic-SSA)
Execution time: 60.0304 seconds
Total Cost: 3234065.50
Furnace Capacity (KF): 2389.21
Installed Machines: [0, 8]

--- Machine 0 ---
Installation Cost: 950000.0
Sections (NSm): 16.0
Gob Type (TGm): 1.0
Efficiency (Nm): 0.9150
Total production on machine 0: 1939.48

--- Machine 8 ---
Installation Cost: 970000.0
Sections (NSm): 8.0
Gob Type (TGm): 2.0
Efficiency (Nm): 0.8990
Total production on machine 8: 449.73
